# 01A 文本、概率与 Loss
对应 **L01.01–L01.04**。先猜结果，再执行；只需 CPU，无需 API。

目标：自己构造下一 Token 样本，计算概率和交叉熵，并解释训练目标的边界。

建议边学边改：每个“练习”先独立完成，再查看“参考解答”。已保存输出来自实际执行，重启内核后可以从上到下重跑。


## 学习路线与配套课件

实践 1A：从文本到概率与损失（`L01-P01`）。

建议先完成本节概念正课，再进入本实践小节。这个 Notebook 可用独立新内核从头运行。先预测，再执行代码、修改一个条件并解释结果。

对应课件稳定编号：L01.01-S01、L01.01-S02、L01.02-S01、L01.02-S02、L01.02-S03、L01.02-S04、L01.02-S05、L01.02-S06、L01.03-S01、L01.03-S02、L01.03-S03、L01.04-S01、L01.04-S02、L01.04-S03、L01.04-S04、L01.04-S05。页数调整不改变这些编号。

按“问题—代码—观察—练习—可复用结果”的顺序学习。先完成练习，再展开参考分析。回放、保存的真实记录和新的在线请求都会明确标注；在线请求默认关闭。

In [1]:
from pathlib import Path
import sys
# 无论从仓库根目录还是 Notebook 目录启动，都定位到同一份课程资料。
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "00-资料库使用说明.md").exists())
CHAPTER = ROOT / "01 大模型基础与最小训练实验"
import torch
from torch.nn import functional as F
torch.manual_seed(7)
torch.set_num_threads(2)
print("PyTorch:", torch.__version__, "；本实验使用 CPU")


PyTorch: 2.8.0+cpu ；本实验使用 CPU


## 语言模型到底预测什么
给定“我喜欢”，下一步可能是“学”，也可能是“看”。语言模型输出整个词表的分布，再由解码程序决定下一个 Token。

**先预测：**生成 3 个新字符需要使用下一 Token 分布多少次？下面用人工指定的分布观察循环，**不是一个训练好的模型**。


In [2]:
vocab_demo = ["学", "看", "书"]
probabilities = torch.tensor([0.6, 0.3, 0.1])
rng = torch.Generator().manual_seed(7)
text = "我喜欢"
for step in range(3):
    chosen = torch.multinomial(probabilities, 1, generator=rng).item()
    text += vocab_demo[chosen]
    print(f"第 {step+1} 次预测与采样: {text}")
# 真正的模型会根据变化的前文重新计算分布；这里故意固定分布以观察生成循环。


第 1 次预测与采样: 我喜欢学
第 2 次预测与采样: 我喜欢学学
第 3 次预测与采样: 我喜欢学学学


## 字符与 Token ID
本次为了读懂代码，把一个字符当一个 Token。真实模型的分词器通常不等于逐字编码，也不保证一个汉字只占一个 Token。

**先预测：**词表只见过“我喜欢学习”，遇到“猫”会发生什么？


In [3]:
text = "我喜欢学习"
vocab = ["<UNK>"] + sorted(set(text))
stoi = {ch: i for i, ch in enumerate(vocab)}
encode = lambda s: [stoi.get(ch, 0) for ch in s]
decode = lambda ids: "".join(vocab[i] for i in ids)
ids = encode(text)
print("词表:", vocab)
print("编码:", ids, "；解码:", decode(ids))
print("未知字符:", encode("猫"), decode(encode("猫")))
assert decode(ids) == text


词表: ['<UNK>', '习', '喜', '学', '我', '欢']
编码: [4, 2, 5, 3, 1] ；解码: 我喜欢学习
未知字符: [0] <UNK>


In [4]:
sequence = torch.tensor(ids)
x, y = sequence[:-1], sequence[1:]
print("输入:", decode(x.tolist()))
print("标签:", decode(y.tolist()))
for i, (a,b) in enumerate(zip(x,y)):
    print(f"位置 {i}: 看见 {decode(x[:i+1].tolist())!r}，目标是 {vocab[b.item()]!r}")
# 每个位置都有目标；标签来自后一个字符，不是人工给整段回答打分。


输入: 我喜欢学
标签: 喜欢学习
位置 0: 看见 '我'，目标是 '喜'
位置 1: 看见 '我喜'，目标是 '欢'
位置 2: 看见 '我喜欢'，目标是 '学'
位置 3: 看见 '我喜欢学'，目标是 '习'


## 先分文档，再切窗口
不能先把同一文本切成大量重叠片段，再随机混进训练与验证两边。下面检查固定文档分割，再观察 [B,T]：B 是 batch 大小，T 是上下文长度。词表只从训练文档建立。


In [5]:
import json
sys.path.insert(0, str(CHAPTER / "代码"))
from tiny_lm import load_data, windows
corpus = json.loads((CHAPTER/"数据/corpus.json").read_text(encoding='utf-8'))
train_texts = {d["text"] for d in corpus if d["split"] == "train"}
valid_texts = {d["text"] for d in corpus if d["split"] == "valid"}
assert not train_texts & valid_texts
vocab_train, train_docs, valid_docs = load_data(CHAPTER/"数据/corpus.json")
xs, ys = windows(train_docs, context=8)
print("文档数:", len(train_docs), len(valid_docs))
print("取两个窗口，输入/标签形状:", xs[:2].shape, ys[:2].shape)
assert torch.equal(xs[:,1:], ys[:,:-1])


文档数: 20 4
取两个窗口，输入/标签形状: torch.Size([2, 8]) torch.Size([2, 8])


## logits 与 softmax
logits 是任意实数分数，softmax 把相对分数变成概率。先减去最大值，能避免指数溢出；不会改变结果。

**先预测：**给所有分数同时加 100，概率会变化吗？


### softmax：公式与小算例

标准公式：$$p_i=\frac{\exp(z_i)}{\sum_j\exp(z_j)}.$$
稳定形式：$$p_i=\frac{\exp(z_i-m)}{\sum_j\exp(z_j-m)},\quad m=\max_j z_j.$$

|候选|分数|减最大值|指数|概率|
|---|---:|---:|---:|---:|
|学|2|0|1.0000|0.6652|
|看|1|−1|0.3679|0.2447|
|书|0|−2|0.1353|0.0900|

先读符号：z 是模型给候选的分数，p 才是概率；i 指某一个候选，求和符号 Σ 表示把全部候选加起来。exp(x) 就是 e 的 x 次方。标准公式先取指数，再除以所有指数的总和。右边是等价的稳定写法：每个分数都减去同一个最大值 m，分子和分母都乘了 exp(−m)，所以概率不变。本例最大值是 2，减完得到 [0, −1, −2]，指数约为 [1, 0.3679, 0.1353]，总和约 1.5032，再逐项相除。请算出第二个候选“看”的概率。显示值经过四舍五入，可能加起来不是恰好 1；未舍入值的和为 1。所有分数同时加 100，减最大值后的数不变，概率也不变。这里只保留三个候选帮助手算，不是模型真实词表或训练输出。下方 prob-example 将验证计算与平移不变性。

参考：https://docs.scipy.org/doc/scipy/reference/generated/scipy.special.softmax.html


In [6]:
logits = torch.tensor([2.0, 1.0, 0.0])
shifted = logits - logits.max()
probs = shifted.exp() / shifted.exp().sum()
print("候选: 学、看、书；概率:", probs.tolist())
assert torch.allclose(probs.sum(), torch.tensor(1.0))
assert torch.allclose(probs, (logits+100).softmax(dim=-1))
print("最大概率下标:", probs.argmax().item())
print("采样下标:", torch.multinomial(probs, 12, replacement=True).tolist())


候选: 学、看、书；概率: [0.6652409434318542, 0.2447284758090973, 0.09003057330846786]
最大概率下标: 0
采样下标: [0, 0, 1, 0, 2, 1, 0, 0, 1, 1, 0, 0]


### 练习：提高目标“看”的概率
只改变第二个 logit，让“看”成为最大概率候选。写出一个可运行答案，并解释为什么“成为最大概率”仍不保证每次采样都选中它。


In [7]:
student_logits = logits.clone()
# TODO：只修改 student_logits[1]，再观察概率。
print(student_logits.softmax(-1))


tensor([0.6652, 0.2447, 0.0900])


### 参考解答
分数 3 大于另外两个分数；概率仍分配给其他候选，随机采样就仍有机会选中它们。


In [8]:
answer_logits = logits.clone()
answer_logits[1] = 3.0
answer_probs = answer_logits.softmax(-1)
assert answer_probs.argmax().item() == 1
assert 0 < answer_probs[1].item() < 1
print(answer_probs.tolist())


[0.2594964802265167, 0.7053845524787903, 0.03511902689933777]


## 单位置 Loss：−ln(目标概率)
目标概率越高，交叉熵越小。使用自然对数，单位可写作 nat。分布中的其他候选通过概率归一化间接影响损失。

**先预测：**正确目标概率从 0.1 提高到 0.9，损失如何变化？


### 单个目标的交叉熵：公式与小算例

单个有效位置（类别 ID 标签，无权重、无标签平滑）：$$\ell=-\ln p_y.$$

同一组 logits 为 `[2,1,0]`。正确目标为“看”（ID 1），因此 $p_y\approx0.2447$，$\ell\approx1.4076$。

`F.cross_entropy(logits[None, :], target)` 直接接收 logits。

沿用 softmax 的三个候选，标签指定正确答案是第二个候选“看”，因此 y=1。计算 Loss 时取 0.2447，而不是最大概率 0.6652。对目标概率取负自然对数得到约 1.4076 nat；使用未舍入概率计算。本页公式针对单个有效位置、类别 ID 标签、无类别权重且无标签平滑。训练目标概率越大，这个损失越小；标签本身对不对要靠数据检查。手算可以先求概率；PyTorch F.cross_entropy 的输入却应是原始 logits，它内部执行稳定计算，不要把 softmax 后的概率再作为 logits 传入。请找到代码中的 target=[1]，再与手算结果对照。

参考：https://docs.pytorch.org/docs/2.14/generated/torch.nn.CrossEntropyLoss.html


In [9]:
p = torch.tensor([0.1, 0.5, 0.9])
for prob, loss in zip(p, -p.log()):
    print(f"目标概率 {prob:.1f} -> Loss {loss:.4f}")
target = torch.tensor([1])  # 正确 Token 为“看”
loss_manual = -probs[target].log().mean()
loss_torch = F.cross_entropy(logits[None,:], target)
assert torch.allclose(loss_manual, loss_torch)
print("同一位置，手算与 PyTorch:", loss_manual.item(), loss_torch.item())
# cross_entropy 接收 logits，不需要先 softmax。


目标概率 0.1 -> Loss 2.3026
目标概率 0.5 -> Loss 0.6931
目标概率 0.9 -> Loss 0.1054
同一位置，手算与 PyTorch: 1.4076058864593506 1.4076058864593506


### 固定目标，看概率和 Loss 一起变化

配套课件：L01.03-S03、L01.04-S01、L01.04-S02。

候选顺序始终是“学、看、书”，正确目标始终是“看”。我们只把“看”的分数从 1 提高到 3。先预测：它的概率会怎样变化？另外两个分数不变，它们的概率会不会也改变？单位置交叉熵为 $L=-\ln p_y$，其中 $y$ 是正确目标的下标。

In [10]:
# 两行是同一位置的两个对照方案，不是两个训练样本。
case_scores = torch.tensor([[2., 1., 0.], [2., 3., 0.]])
case_prob = torch.softmax(case_scores, dim=-1)
case_loss = F.cross_entropy(case_scores, torch.tensor([1, 1]), reduction="none")
for label, row, loss_value in zip(["修改前", "修改后"], case_prob, case_loss):
    print(label, "三项概率:", [round(v, 4) for v in row.tolist()],
          "目标概率:", round(row[1].item(), 4), "Loss:", round(loss_value.item(), 4))
assert case_prob[1, 1] > case_prob[0, 1] and case_loss[1] < case_loss[0]
assert torch.allclose(case_loss, -case_prob[:, 1].log())


修改前 三项概率: [0.6652, 0.2447, 0.09] 目标概率: 0.2447 Loss: 1.4076
修改后 三项概率: [0.2595, 0.7054, 0.0351] 目标概率: 0.7054 Loss: 0.349


**结果解读**

目标概率约从 0.2447 升到 0.7054，Loss 约从 1.4076 降到 0.3490。其他候选的概率下降，因为 softmax 的分母也变了。这里手工改了分数；训练则通过更新模型参数改变分数。

**小练习**

只改第二行，把“书”的分数提高到 3，仍把“看”作为目标。预测 Loss 会升还是降，再验证。

<details><summary>完成后展开参考分析</summary>

目标“看”仍为 1，竞争候选升高会压低它的概率，Loss 上升。最大概率候选不一定是正确目标。

</details>

公式与实现来源见 [配套素材来源](../../资料来源.md)。

## 从单位置到多个位置
下面每行对应一个目标位置。先分别计算负对数概率，再对有效位置求平均。遇到 padding 时要明确哪些位置不参与损失，本课固定窗口没有 padding。


### 平均 Loss：公式与小算例

$$L=\frac{1}{n}\sum_{t=1}^{n}\ell_t,\qquad \ell_t=-\ln p_{t,y_t}.$$

两个位置手算：$(-\ln0.5-\ln0.25)/2\approx1.0397$。先平均概率再取对数为 $-\ln((0.5+0.25)/2)\approx0.9808$，二者不同。下方代码用另一组三位置样本，不能把这里的数字当成它的运行输出。

先对每个有效位置计算损失，再把损失求平均。本页是额外的两个位置手算练习，用来展示运算顺序，不是下方三位置 batch_logits 的输出。两个目标概率为 0.5、0.25，损失分别约 0.6931、1.3863，平均约 1.0397。若先平均概率，得到 0.375，再取负对数是约 0.9808，因此两个顺序不等价。此处没有类别权重，也没有 padding；有忽略位置时，只对参与损失计算的位置计数。接着在 loss-mean 看实际三个位置的结果，它们的目标概率相同，所以每项损失均约 0.4076，平均仍为 0.4076。请先解释为什么，再运行。

参考：https://docs.pytorch.org/docs/2.14/generated/torch.nn.CrossEntropyLoss.html


In [11]:
batch_logits = torch.tensor([[2.,1.,0.], [0.,2.,1.], [1.,0.,2.]])
targets = torch.tensor([0,1,2])
per_position = F.cross_entropy(batch_logits, targets, reduction="none")
print("逐位置:", per_position.tolist(), "平均:", per_position.mean().item())
assert torch.allclose(per_position.mean(), F.cross_entropy(batch_logits,targets))
# 标签如果错误，优化仍会鼓励它。用第一个位置体验这一点。
wrong_target_loss = F.cross_entropy(batch_logits[:1], torch.tensor([2]))
print("把标签换成第三个 Token，Loss:", wrong_target_loss.item())


逐位置: [0.40760594606399536, 0.4076060354709625, 0.40760594606399536] 平均: 0.40760597586631775
把标签换成第三个 Token，Loss: 2.4076058864593506


### 练习：平均 Loss 的实现
输入是每个位置的正确目标概率。请实现平均负对数概率。思考：如果把概率直接相加再取负对数，会得到同一个值吗？


In [12]:
def student_loss(target_probs):
    # TODO：返回一个标量张量。完成后与下方参考实现对比。
    return None
print("你的结果:", student_loss(torch.tensor([0.5, 0.25])))


你的结果: None


In [13]:
def mean_nll(target_probs):
    if torch.any((target_probs <= 0) | (target_probs > 1)):
        raise ValueError("目标概率必须位于 (0,1]")
    return -target_probs.log().mean()
answer = mean_nll(torch.tensor([0.5,0.25]))
assert abs(answer.item() - 1.03972077) < 1e-6
print("平均 Loss:", answer.item())


平均 Loss: 1.0397207736968994


## 提交与自检
提交一份修改过的 logits、对应概率、一个平均 Loss 的计算和解释。回答：

1. 为什么输入与标签只错开一位？
2. 为什么训练 Loss 很低仍不能证明回答事实正确？
3. 如果验证文档大量含 `<UNK>`，验证 Loss 的解释有什么局限？

接着打开 **02-causal-transformer.ipynb**。当前实验还没有可训练的语言模型。
